<a href="https://colab.research.google.com/github/taahahussain/KhateebGPT/blob/main/Copy_of_KhateebGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from transformers import TrainingArguments, Trainer, BertTokenizer, BertForSequenceClassification
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity



In [4]:
df = pd.read_csv('all_hadiths_clean.csv')
df

,id,hadith_id,source,chapter_no,hadith_no,chapter,chain_indx,text_ar,text_en
0,0,1,Sahih Bukhari,1,1,Revelation - كتاب بدء الوحى,"30418, 20005, 11062, 11213, 11042, 3",حدثنا الحميدي عبد الله بن الزبير، قال حدثنا سف...,Narrated 'Umar bin Al-Khattab: ...
1,1,2,Sahih Bukhari,1,2,Revelation - كتاب بدء الوحى,"30355, 20001, 11065, 10511, 53",حدثنا عبد الله بن يوسف، قال أخبرنا مالك، عن هش...,Narrated 'Aisha: ...
2,2,3,Sahih Bukhari,1,3,Revelation - كتاب بدء الوحى,"30399, 20023, 11207, 11013, 10511, 53",حدثنا يحيى بن بكير، قال حدثنا الليث، عن عقيل، ...,Narrated 'Aisha: (the m...
3,3,4,Sahih Bukhari,1,4,Revelation - كتاب بدء الوحى,"11013, 10567, 34",قال ابن شهاب وأخبرني أبو سلمة بن عبد الرحمن، أ...,Narrated Jabir bin 'Abdullah Al-Ansari while ...
4,4,5,Sahih Bukhari,1,5,Revelation - كتاب بدء الوحى,"20040, 20469, 11399, 11050, 17",حدثنا موسى بن إسماعيل، قال حدثنا أبو عوانة، قا...,Narrated Said bin Jubair: ...
...,...,...,...,...,...,...,...,...,...
34436,173,54223,Sunan Ibn Majah,36,4234,Tribulations - كتاب الفتن,"30285, 20733, 20399, 11412, 11027, 3051",حدثنا عبد الرحمن بن إبراهيم، حدثنا الوليد بن م...,It was narrated from ‘Awf bin Malik Al-Ashja’...
34437,174,54224,Sunan Ibn Majah,36,4235,Tribulations - كتاب الفتن,"30201, 20005, 11013, 11002",حدثنا أبو بكر بن أبي شيبة، حدثنا سفيان بن عيين...,It was narrated from Abu Hurairah conveying i...
34438,175,54225,Sunan Ibn Majah,36,4236,Tribulations - كتاب الفتن,"30201, 20005, 11061, 11197, 13",حدثنا أبو بكر بن أبي شيبة، حدثنا سفيان بن عيين...,It was narrated from Abu Hurairah that the Me...
34439,176,54226,Sunan Ibn Majah,36,4237,Tribulations - كتاب الفتن,"30201, 20380, 11272, 11016, 5200",حدثنا أبو بكر بن أبي شيبة، حدثنا أسود بن عامر،...,It was narrated that ‘Amr bin Taghlib said: ...


In [5]:
from transformers import BertTokenizer, BertForSequenceClassification
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=2)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
model = model.to('cuda')

In [7]:
!pip install -U sentence_transformers

  Using cached sentence_transformers-3.0.1-py3-none-any.whl (227 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86

In [8]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
df = df['text_en']
first_15 = df.head(15)
first_15

0           Narrated 'Umar bin Al-Khattab:          ...
1           Narrated 'Aisha:                        ...
2      Narrated 'Aisha:                       (the m...
3      Narrated Jabir bin 'Abdullah Al-Ansari while ...
4           Narrated Said bin Jubair:               ...
5           Narrated Ibn 'Abbas:                    ...
6           Narrated 'Abdullah bin 'Abbas:          ...
7           Narrated Ibn 'Umar:                     ...
8           Narrated Abu Huraira:                   ...
9           Narrated 'Abdullah bin 'Amr:            ...
10          Narrated Abu Musa:                      ...
11          Narrated 'Abdullah bin 'Amr:            ...
12          Narrated Anas:                          ...
13          Narrated Abu Huraira:                   ...
14          Narrated Anas:                          ...
Name: text_en, dtype: object

In [10]:
df_15_rand = df.sample(15)
df_15_rand

11134     Ibn 'Umar (Allah be pleased with them) said t...
12011     The tradition has been narrated by a differen...
27317     It was narrated from Jabir that:             ...
27610     It was narrated from Kathir bin Murrah that t...
935       Narrated Ibn Juraij:                     `Ata...
33712     It wasnarrated that Ibn ‘Abbas said:         ...
3445      Narrated `Adi bin Hatim:                     ...
17939     Narrated Abdullah ibn Abbas:                 ...
23071     Narrated Shu'bah:                     "From A...
8456      Ibn Abbas reported:                      When...
34200     It was narrated from Abu Na'amah that :      ...
34086     It was narrated from Abu Malih AL-HUdhail tha...
15555     Narrated Abdullah ibn Umar:                  ...
21908     Narrated Yazid bin Abu Maryam:               ...
9914      Abdullah b. 'Umar (Allah be pleased with both...
Name: text_en, dtype: object

In [11]:
#Encode all sentences
embeddings = model.encode(first_15.tolist())

#Compute cosine similarity between all pairs
cos_sim = util.cos_sim(embeddings, embeddings)

len(cos_sim)

15

In [12]:
all_hadith_combination = []
for i in range(len(cos_sim) - 1):
    for j in range(i+1, len(cos_sim)):
        all_hadith_combination.append((cos_sim[i][j], i, j))
all_hadith_combination

[(tensor(0.1116), 0, 1),
 (tensor(0.1493), 0, 2),
 (tensor(0.0756), 0, 3),
 (tensor(0.1018), 0, 4),
 (tensor(0.2698), 0, 5),
 (tensor(0.1554), 0, 6),
 (tensor(0.1635), 0, 7),
 (tensor(0.0508), 0, 8),
 (tensor(0.2786), 0, 9),
 (tensor(0.0992), 0, 10),
 (tensor(0.3338), 0, 11),
 (tensor(0.2827), 0, 12),
 (tensor(0.2753), 0, 13),
 (tensor(0.2506), 0, 14),
 (tensor(0.4372), 1, 2),
 (tensor(0.5081), 1, 3),
 (tensor(0.3800), 1, 4),
 (tensor(0.3235), 1, 5),
 (tensor(0.2307), 1, 6),
 (tensor(0.2274), 1, 7),
 (tensor(0.3381), 1, 8),
 (tensor(0.1590), 1, 9),
 (tensor(0.2290), 1, 10),
 (tensor(0.2919), 1, 11),
 (tensor(0.3325), 1, 12),
 (tensor(0.3817), 1, 13),
 (tensor(0.3746), 1, 14),
 (tensor(0.4985), 2, 3),
 (tensor(0.4346), 2, 4),
 (tensor(0.2109), 2, 5),
 (tensor(0.3435), 2, 6),
 (tensor(0.2139), 2, 7),
 (tensor(0.2558), 2, 8),
 (tensor(0.2713), 2, 9),
 (tensor(0.2903), 2, 10),
 (tensor(0.3450), 2, 11),
 (tensor(0.3434), 2, 12),
 (tensor(0.3903), 2, 13),
 (tensor(0.3976), 2, 14),
 (tensor(0

In [14]:
!pip install keybert

  Using cached keybert-0.8.5-py3-none-any.whl (37 kB)


In [18]:
from keybert import KeyBERT
kw_model = KeyBERT(model='all-MiniLM-L6-v2')
keywords = []

# Iterate over the values in the Series using .iteritems()
for index, row in first_15.items():
    kw = kw_model.extract_keywords(row, keyphrase_ngram_range=(1, 3), stop_words='english', top_n=10)
    keywords.append(kw)
keywords

[[('marry emigration emigrated', 0.6768),
  ('emigrated worldly benefits', 0.6546),
  ('marry emigration', 0.6519),
  ('emigration emigrated', 0.6291),
  ('woman marry emigration', 0.59),
  ('according intended emigrated', 0.5766),
  ('emigration', 0.5762),
  ('intended emigrated worldly', 0.5639),
  ('intended emigrated', 0.5598),
  ('emigrated worldly', 0.4908)],
 [('divine inspiration revealed', 0.7279),
  ('inspiration revealed allah', 0.6781),
  ('prophet inspired divinely', 0.6745),
  ('divine inspiration', 0.673),
  ('inspired divinely', 0.649),
  ('prophet inspired', 0.6218),
  ('inspiration revealed', 0.6209),
  ('inspired divinely cold', 0.5979),
  ('inspired angel', 0.5918),
  ('saw prophet inspired', 0.5854)],
 [('read prophet', 0.5658),
  ('asked read prophet', 0.546),
  ('khadija replied allah', 0.5302),
  ('know read prophet', 0.5237),
  ('read prophet replied', 0.5155),
  ('khadija said', 0.5033),
  ('khadija accompanied', 0.5001),
  ('khadija replied', 0.489),
  ('khad

In [17]:
kw_model = KeyBERT()

# Provided text
text = """
It is narrated on the authority of Amir al-Mu'minin (Leader of the Believers), Abu Hafs 'Umar bin al-Khattab (may Allah be pleased with him), who said: I heard the Messenger of Allah (peace be upon him), say

"Actions are according to intentions, and everyone will get what was intended. Whoever migrates with an intention for Allah and His messenger, the migration will be for the sake of Allah and his Messenger. And whoever migrates for worldly gain or to marry a woman, then his migration will be for the sake of whatever he migrated for."
"""

# Extract keywords
keywords = kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words='english', top_n=10)

# Print the extracted keywords
for keyword in keywords:
    print(keyword)

('migrates intention allah', 0.7335)
('allah messenger migration', 0.6577)
('migrates intention', 0.6143)
('intended migrates intention', 0.6129)
('intentions intended migrates', 0.6089)
('allah messenger migrates', 0.5972)
('migration sake allah', 0.5853)
('marry woman migration', 0.5842)
('woman migration sake', 0.567)
('woman migration', 0.5639)
